# Digits parameter sweeps

Explore how `IteratedMSTEmbedding` parameters change the digits embedding. The sweep below defaults to the documented MST counts (`1, 2, 4, 8, 16, 32`), uses the same seed for each run, and reports trustworthiness alongside the plots.

The default 500 epochs and larger MST counts can take a few minutes. For a quick iteration, lower `N_EPOCHS` or shorten `VALUES`; restore 500 epochs for a final comparison.

In [ ]:
from pathlib import Path
import sys

# Add the notebook helper module when running from the repository or notebooks/.
repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "digits_sweep.py").is_file()),
    None,
)
if repo_root is not None:
    sys.path.insert(0, str(repo_root / "notebooks"))

from digits_sweep import load_digits_data, run_sweep

## Load and scale digits

In [ ]:
X, labels = load_digits_data()
print(f"Samples: {X.shape[0]}, features: {X.shape[1]}")

## Choose a sweep

Set `PARAMETER` to any estimator parameter, such as `"n_msts"`, `"lambda_rep"`, or `"learning_rate"`, and set `VALUES` to the candidates. `BASE_PARAMS` supplies the remaining estimator settings.

Set `N_JOBS=1` for sequential runs or `N_JOBS=2` to fit two candidates at once. Each worker builds a dense pairwise distance matrix, so keep the worker count modest for larger datasets.

In [ ]:
PARAMETER = "n_msts"
VALUES = [1, 2, 4, 6, 8, 10]

BASE_PARAMS = {
    "n_epochs": 500,
    "batch_size": 4096,
    "learning_rate": 0.05,
    "negative_ratio": 4,
    "lambda_rep": 1.0,
    "random_state": 42,
    "epsilon": 1e-4,
}
TRUSTWORTHINESS_NEIGHBORS = 10
N_JOBS = 2

Each panel colors samples by their digit label and reports 5-fold cross-validated 5-nearest-neighbor accuracy using the 2D coordinates. Trustworthiness summarizes how well local neighborhoods in the original standardized data are preserved in 2D. Both scores are for comparison only and are not used to train the embedding. The accuracy is a transductive estimate: the embedding was fitted jointly on all unlabeled samples before the classifier folds were evaluated.

In [ ]:
results, summary = run_sweep(
    X, labels, PARAMETER, VALUES, BASE_PARAMS,
    n_neighbors=TRUSTWORTHINESS_NEIGHBORS, n_jobs=N_JOBS,
)
summary

## Inspect one result

The full coordinates are retained in `results`; for example, select an embedding by its sweep index and reuse it in your own plots or downstream analysis.

In [ ]:
# Example: first embedding from the sweep
first_embedding = results[0]["embedding"]
first_embedding.shape